# Ohùn: export + publish the Yoruba voice (CPU only)

Attach the training notebook's output (`ohun-yoruba-train`) as input and attach the `HF_TOKEN` secret. No GPU needed. Exports the newest `last.ckpt` to ONNX (legacy exporter, `dynamo=False`) and uploads `ohun_yo.onnx`, `ohun_yo.onnx.json` and `checkpoints/last.ckpt` to `aladewealth/ohun-yoruba`.

In [ ]:
!git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git || true
%cd /kaggle/working/piper1-gpl
!sed -i 's/verbose=False,/verbose=False, dynamo=False,/' src/piper/train/export_onnx.py && grep -n 'dynamo=False' src/piper/train/export_onnx.py
!pip install -q scikit-build
!pip install -q -e '.[train]'
!pip install -q onnx onnxruntime huggingface_hub
!./build_monotonic_align.sh && python3 setup.py build_ext --inplace -q
%cd /kaggle/working

In [ ]:
import os, glob, subprocess
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
W = '/kaggle/working'; REPO = 'aladewealth/ohun-yoruba'
ck = sorted(glob.glob('/kaggle/input/**/last.ckpt', recursive=True), key=os.path.getmtime)
js = glob.glob('/kaggle/input/**/ohun_yo.onnx.json', recursive=True)
print(ck, js)
CKPT = ck[-1]
import shutil; shutil.copy(js[0], f'{W}/ohun_yo.onnx.json')
r = subprocess.run(['python3','-m','piper.train.export_onnx','--checkpoint',CKPT,'--output-file',f'{W}/ohun_yo.onnx'],
                   cwd=f'{W}/piper1-gpl', capture_output=True, text=True, env=os.environ.copy())
print(r.stdout[-2000:], r.stderr[-4000:])
assert os.path.exists(f'{W}/ohun_yo.onnx'), 'export failed'
print('exported', os.path.getsize(f'{W}/ohun_yo.onnx')/1e6, 'MB')

In [ ]:
try:
    # sanity check: run the ONNX once
    import onnxruntime as ort, numpy as np, json
    s = ort.InferenceSession(f'{W}/ohun_yo.onnx')
    cfg = json.load(open(f'{W}/ohun_yo.onnx.json'))
    ids = np.array([[1, 3, 4, 5, 6, 7, 8, 9, 10, 2]], dtype=np.int64)
    feed = {'input': ids, 'input_lengths': np.array([ids.shape[1]], dtype=np.int64), 'scales': np.array([0.667,1.0,0.8], dtype=np.float32)}
    if 'sid' in [i.name for i in s.get_inputs()]: feed['sid'] = np.array([0], dtype=np.int64)
    out = s.run(None, feed)[0]; print('audio samples', out.size, 'sr', cfg['audio']['sample_rate'])
except Exception as e:
    print('sanity check skipped:', repr(e))

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
for f in ['ohun_yo.onnx','ohun_yo.onnx.json']:
    api.upload_file(path_or_fileobj=f'{W}/{f}', path_in_repo=f, repo_id=REPO, commit_message='Ohùn Yoruba weights')
    print('uploaded', f)
api.upload_file(path_or_fileobj=CKPT, path_in_repo='checkpoints/last.ckpt', repo_id=REPO, commit_message='Ohùn training checkpoint (resume point)')
print('uploaded checkpoints/last.ckpt  ->  https://huggingface.co/' + REPO)